# 04 · Baseline, read the errors, iterate, freeze

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/egumasa/lda2-final-template/blob/main/notebooks/04_prompt.ipynb)

Write the plainest prompt that could work, then improve it for reasons you can state.

```
  01_build_pool_<track>  →  02_sample  →  02b_add_samples  →  03_annotate  →▶ 04_prompt  →  05_report
```

| | |
|---|---|
| **Reads** | `data/gold/<track>_<group>_dev.json` (from 03) · the pool (from 01) |
| **Writes** | `outputs/<track>_<group>_predictions.json` · `..._rounds.json` · `..._test_log.jsonl` |

---

Everything from here on is measured against **your** gold set, not the corpus's labels. That is the point of the last two notebooks.

Steps 1–3 work on your **dev** half. The test half is not opened until step 4, and then only once.

> **Free-tier pacing.** The backend waits a few seconds between calls and retries on rate-limit errors, so a run takes minutes and may print `(rate limited - waiting Ns then retrying)`. That is normal — and it is why you iterate on dev: a dozen or so items is about a minute per round, so you get enough rounds to actually learn something. Your sample stays at full size throughout; there is no longer a small-while-you-iterate phase.

> **On the size of this study.** One call per item, four-and-a-bit seconds apart, no batching: forty items is minutes and four hundred is most of an afternoon of a quota you share with everyone else on the course. A study that could support a claim about a corpus needs hundreds of items per class. This one cannot, and that is a limitation to state in report §5 rather than write around. What transfers is the method — the split, the freezing, the audit trail — not the number.

## Setup — run this first

This cell mounts your Google Drive and finds your group's shared folder, `lda2-final-template`. Everything the project produces — the pool, the gold set, your prompts, the outputs — is an ordinary file in there, which is what makes it survive the runtime resetting *and* lets the rest of your group see it.

**One member sets the folder up once:**

1. That member runs the `git clone` line this cell prints if the folder is missing, which puts it in their own Drive.
2. They share it with the group (right-click ▸ *Share*), with edit access.
3. Everyone else opens *Shared with me*, right-clicks the folder, and chooses **Add shortcut to Drive** ▸ *My Drive*.

Keep that shortcut's name exactly `lda2-final-template`. It is what makes the same path work for all of you — if Drive renames it to `lda2-final-template (1)`, this cell will not find it.

From then on, open notebooks from the folder itself (*File ▸ Open notebook ▸ Drive*) rather than from the GitHub badge, so you are working on your group's copy and not a fresh one.

**Looking inside a helper.** The functions this cell imports are defined in `scripts/`. Two ways to read one, both the same ones you used on Day 2:

- `help(save_json)` prints its first line — what to pass in and what comes back — and the description of each argument. Typing `save_json(` and pressing **Shift+Tab** shows the same thing in a pop-up.
- To read the code itself, open `scripts/pipeline.py` from the **Files** panel on the left. Colab lists the functions in that file down the side, so you can click straight to the one you want.

In [ ]:
# ------------------------------------------------------------------
# SETUP — run me first. You are not expected to read it.
# ------------------------------------------------------------------
# This cell is plumbing, and it is the only cell in the project that is.
# It finds your group's shared folder in Google Drive, because everything
# this project keeps goes in there: a Colab runtime is wiped when it resets,
# and nobody else in your group can see inside it. Then it makes the
# project's own code importable. Run it and move on; nothing below asks you
# to have understood it.

FOLDER = "lda2-final-template"     # the shared folder, in every member's Drive

import os, sys

PROJECT = ".."                              # running locally: it is just above us

try:
    from google.colab import drive           # only exists inside Colab
except ImportError:
    pass
else:
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/" + FOLDER
    if not os.path.isdir(PROJECT):
        raise RuntimeError(
            "Could not find " + PROJECT + "\n\n"
            "Setting the folder up for your group? Run this in a new cell:\n"
            "  !git clone https://github.com/egumasa/lda2-final-template.git "
            + PROJECT + "\n"
            "then share the folder with the rest of your group.\n\n"
            "Someone else already did? Open Drive, find the folder under "
            "'Shared with me', right-click it, and choose 'Add shortcut to "
            "Drive'. Keep the name exactly " + FOLDER + ".")
    # Work inside the project folder, where the notebooks live.
    os.makedirs(PROJECT + "/notebooks", exist_ok=True)
    os.chdir(PROJECT + "/notebooks")

# scripts/ and config.py, by their real paths - so they are found from wherever
# this notebook happens to be working.
sys.path.append(PROJECT)
sys.path.append(PROJECT + "/scripts")

# Re-read config.yaml every time this cell runs. Without the reload, Python
# hands back the settings it read the FIRST time, and editing config.yaml
# would appear to do nothing until you restarted the runtime.
import importlib
import config
importlib.reload(config)

# Named one by one rather than with `import *`, so that every name a cell
# below uses can be traced back to the file it came from — config.yaml for
# these, scripts/ for the rest.
from config import (TRACK, GROUP, RUN, SEED, N_PER_CLASS, DEV, CODERS,
                    MEMBERS, LABELS_ORDER, ROOT, OUT_DIR,
                    POOL_PATH, DEMO_POOL_PATH, SAMPLE_PATH, GOLD_PATH,
                    SAMPLE_BEFORE_TOPUP_PATH,
                    DEV_PATH, TEST_PATH, DISAGREED_PATH, PRED_PATH,
                    ROUNDS_PATH, TESTLOG_PATH,
                    PROMPT_FILE, SHEET_PATH, TRIAGE_PATH, describe)

# Files in, files out, and the connection to the model: all plumbing.
from pipeline import (load_gold, label_set, load_prompt, save_json,
                      load_predictions, setup, freeze_test_run)

# Asking the model and scoring the answers is what this notebook is FOR, so
# those five functions are not imported here at all. You define them yourself,
# in step 2, in cells you can read and change, just before the first round.

describe()                  # what this notebook is working on


## Connect to the model

Now we open the connection this notebook will send every prompt through. It gets a cell of its own because it does something the plumbing cell above does not: it reaches out to a service, and what it prints back is worth reading.

Safe to run more than once — after the first time it hands back the connection it already made.

In [ ]:
setup()

> **Check the backend line it just printed.** You want:
>
> ```
> LLM backend: Gemini API (gemini-3.1-flash-lite, temperature=0, seed=42)
> ```
>
> If it says **Colab Gemini** instead, no API key was found — put yours in the Colab Secrets panel (the 🔑 icon in the left sidebar) as `GEMINI_API_KEY` and re-run. The keyless backend has no temperature or seed, so the same prompt can give different answers and your numbers will not be reproducible. It must not be your final run.

> **Everything above comes from `config.yaml`** — one small file at the top of the repo, which you edit once as a group, and the only file in the plumbing you touch. That is deliberate: the seed that drew your sample has to be the seed you report, and five copies of a number in five notebooks is five chances for them to disagree. Your settings are also the filenames — `track: cars50`, `group: kimura`, `run: v1` means this notebook reads and writes `cars50_kimura_v1_...`. If the line it just printed is not your track, your group and your seed, fix `config.yaml` and re-run this cell.

## Step 1 — Open the three files this notebook works from

Now we load the dev half (what you may look at), the pool (where few-shot examples come from) and the **full** gold set.

That third one looks redundant and is not. `build_fewshot` excludes your gold items from the examples it picks — by *text*, since sampling renumbered the ids. Hand it only `dev` and it can pick a **test** item as a worked example, which puts the answer to a held-out item straight into the prompt that produces your headline number. So the full gold set is loaded as an exclusion list, and scored against never.

`LABELS` is read off the full gold set for the same reason: a label that happens to be thin in dev should not quietly shrink your label list.

`TEST_PATH` is not opened here. It is opened once, in step 4. All three of these are the `load_gold` call from the Day 3 setup.

In [ ]:
# ══ STEP 1 · Load your dev set and your pool ══════════════════════════════
# Opens the three files this notebook works from, and reads your label list
# off the full gold set.
# Creates: dev, pool, gold, LABELS

# ✏️ this runs as written — the work is deciding whether it should

dev  = load_gold(DEV_PATH)      # what you iterate against, from notebook 03
pool = load_gold(POOL_PATH)     # the spares few-shot examples are drawn from
gold = load_gold(GOLD_PATH)     # ALL of it — as an EXCLUSION list, see above
LABELS = label_set(gold)        # gold, not pool: what you actually adjudicated

print(len(dev), "dev ·", len(pool), "pool ·", LABELS)


## Step 2 — The baseline (round 0)

Your first score, before you have changed anything. Write the plainest prompt that states the task and the label set, run it, score it. **Resist the urge to make it good** — later rounds need something to be measured against, and a baseline you already tuned tells you nothing about whether tuning helped.

Your prompt lives in `prompts/<track>.txt` and must contain `{text}`, where each item gets slotted in. Edit the **file**, not a string in this notebook — that is what makes each version savable and comparable, and it is the reproducibility habit from S10.

In Colab you can write the file straight from a cell:

```python
%%writefile ../prompts/raamove_v0.txt
Classify the rhetorical move of the sentence. Answer with the move name only.
...

Sentence: {text}
```

The three cells below are the Day 3 Part A run, plus the error table. Every number in this step and the next is a **dev** number: you use it to decide what to change next, not to report.

`f1_by_round` collects one score per round, keyed by the round's name. Notebook 05 reads it back from a file and prints it into your report, so those keys are what your reader sees — name them so they mean something.

In [ ]:
# ══ STEP 2 · Baseline prompt (round 0) ════════════════════════════════════
# Starts the table of per-round scores, and loads the starting prompt for your
# track so you can read it before it runs.
# Creates: f1_by_round, PROMPT

# ✏️ this runs as written — the work is deciding whether it should

# One entry per round from here on. Notebook 05 turns it into your table.
f1_by_round = {}

PROMPT = load_prompt(PROMPT_FILE)        # the starting prompt for your track
print(PROMPT)


### The five functions that produce your numbers

Every number in your report comes out of these five, so read them before you quote them. SETUP did not import them: **the cells below are where they come from**, so run them before the steps that use them. They are read out of `scripts/` when this notebook is generated — not a simplified copy.

They are ordinary definitions. Change one, run the cell again, and the rounds below use your version. That is worth knowing about `extract_label` in particular: if your model keeps answering in a shape it cannot read, this is where you would fix that.

Five cells is more reading than the earlier notebooks asked for. Take them one at a time — each one runs on its own, and none of them calls the model.

First, the helpers the definitions below call. Nothing to decide here — run it and read on.

In [ ]:
import pandas as pd
import random
import re
from metrics import classification_report, cohen_kappa_score, confusion_matrix, f1_score, label_set, plot_confusion_matrix
from pipeline import _default_backend, label_set

**`show_errors` is the one you will actually iterate on.** In: your dev items and the model's predictions. Out: a `DataFrame` of just the rows where the two differ — which is why you can filter it with `errors[errors.gold == "…"]`. F1 tells you *whether* a round helped; only the errors tell you *what to change next*.

In [ ]:
def show_errors(gold: list[dict[str, str]],
                predictions: list[str]) -> pd.DataFrame:
    """The items the model got wrong, as a table you can read and argue about.

    Args:
        gold: the gold items, each with "id", "text" and "label".
        predictions: one predicted label per gold item, in the same order.

    Returns:
        A table with one row per mistake: id, gold, pred, text. The columns are
        named even when there are no mistakes.

    Example:
        >>> errors = show_errors(gold, predictions)
    """
    rows = []
    for item, predicted in zip(gold, predictions):
        if item["label"] != predicted:
            row = {
                "id": item["id"],
                "gold": item["label"],
                "pred": predicted,
                "text": item["text"],
            }
            rows.append(row)
    print(f"{len(rows)} of {len(gold)} wrong.")
    # Name the columns even when there are no rows. A table built from an empty list
    # has no columns at all, and then errors["gold"] in notebook 05 fails for the one
    # group whose model got everything right - the least deserving group to break on.
    return pd.DataFrame(rows, columns=["id", "gold", "pred", "text"])

**`extract_label` is doing more than it looks.** In: one reply from the model, in prose. Out: one label. This is what decides that *"This looks like Move 2 to me"* means `Move 2` — it searches for label names, keeps the longest match, and falls back to `"??"`. Every `??` in your run is a reply it could not read, and if there are many, that is a finding about your prompt, not a bug.

In [ ]:
def extract_label(reply: str, labels: list[str]) -> str:
    """Figure out which of the known labels the model's reply is pointing at.

    Args:
        reply: whatever the model replied.
        labels: the labels your scheme allows.

    Returns:
        The label it found - the longest one when several appear - or "??" when the
        reply contains none of them.

    Example:
        >>> extract_label("I would say B2.", LEVELS)
    """
    reply_text = str(reply).strip()
    reply_lowercased = reply_text.lower()

    # Step 1: collect every known label whose name appears in the reply.
    labels_found = []
    for label in labels:
        if label.lower() in reply_lowercased:
            labels_found.append(label)

    # Step 2: if we found one or more, keep the longest (most specific) one.
    if len(labels_found) > 0:
        longest_label = labels_found[0]
        for label in labels_found:
            if len(label) > len(longest_label):
                longest_label = label
        return longest_label

    # Step 3: special case for "Move 1/2/3" labels - look for a bare digit.
    has_move_labels = False
    for label in labels:
        if label.lower().startswith("move "):
            has_move_labels = True
    if has_move_labels:
        match = re.search(r"\b([1-9])\b", reply_text)
        if match is not None:
            candidate = "Move " + match.group(1)
            if candidate in labels:
                return candidate

    # Step 4: nothing matched.
    return "??"

**`run_prompt` is the loop.** In: your prompt and a list of items. Out: one predicted label per item. One API call each, the reply passed through `extract_label`. The pacing and retrying happen inside `_default_backend`, which is the connection `setup()` opened — that part is plumbing, and it stays imported.

In [ ]:
def run_prompt(prompt: str,
               gold: list[dict[str, str]],
               labels: list[str] | None = None,
               generate_text=None) -> list[str]:
    """Ask the model to label every item, and collect the predicted labels.

    Same call as Day 3: run_prompt(PROMPT, gold). The two optional arguments are
    worked out for you, so you only pass them if you want something different.

    Args:
        prompt: your prompt, containing {text} where the sentence should go, and
            {context} for its passage on the tracks that carry one.
        gold: the items to label.
        labels: the labels your scheme allows. Left out, they are read off `gold`.
        generate_text: the function that sends a prompt. Left out, the backend
            connected by the Setup cell is used.

    Returns:
        One predicted label per gold item, in the same order. Replies no label
        could be read out of come back as "??".

    Example:
        >>> predictions = run_prompt(prompt, dev)
    """
    if labels is None:
        labels = label_set(gold)
    if generate_text is None:
        generate_text = _default_backend()

    # A prompt that asks for {context} on a track whose items have none would quietly
    # send the model an empty passage, once per item, and report a number as if it had
    # tested something. Say so instead.
    if "{context}" in prompt and not any(item.get("context") for item in gold):
        print("WARNING: this prompt uses {context}, but none of these items carry one. "
              "Only the rhetorical-move tracks (cars50, raamove) do. The model is about "
              "to be shown an empty passage " + str(len(gold)) + " times.")

    predictions = []
    total = len(gold)
    position = 0
    for item in gold:
        position = position + 1
        # Put this item's sentence into the prompt where {text} is - and its passage
        # where {context} is, on the tracks that carry one. A prompt that does not
        # mention {context} simply ignores it.
        filled_prompt = prompt.format(text=item["text"],
                                      context=item.get("context", ""))
        reply = generate_text(filled_prompt)
        predicted_label = extract_label(reply, labels)
        predictions.append(predicted_label)
        # Print a small progress note every 10 items.
        if position % 10 == 0:
            print("  ...", position, "/", total, "done")

    # Count how many replies we could not turn into a valid label.
    number_unparseable = 0
    for label in predictions:
        if label == "??":
            number_unparseable = number_unparseable + 1
    print("Got", len(predictions), "predictions (", number_unparseable, "could not be parsed).")
    return predictions

**`build_fewshot` puts worked examples in front of the model.** In: your prompt and the pool. Out: the same prompt with a few solved items added to it. It skips anything in your gold set — matched by text, because sampling renumbered the ids. Without that skip you would be testing the model on answers you had just shown it. You use it in step 3.

In [ ]:
def build_fewshot(base_prompt: str,
                  pool: list[dict[str, str]],
                  gold: list[dict[str, str]],
                  labels: list[str] | None = None,
                  shots_per_class: int = 1,
                  seed: int = 42) -> str:
    """Put a few labeled examples (taken from the pool) in front of the prompt.

    We NEVER use an item that is in the gold set as an example, otherwise we
    would be showing the model the very answers we are testing it on. Items are
    matched by their TEXT, not their id, because sampling renumbers the ids.

    Args:
        base_prompt: the prompt to put the examples in front of.
        pool: the items to draw examples from.
        gold: the items being tested. None of these is used as an example.
        labels: the labels to find examples for. Left out, read off `gold`.
        shots_per_class: how many examples to show for each label.
        seed: same seed gives the same examples.

    Returns:
        The example block followed by your prompt.

    Example:
        >>> prompt = build_fewshot(base_prompt, pool, gold, shots_per_class=2)
    """
    if labels is None:
        labels = label_set(gold)

    # Step 1: collect the texts that are already in the gold set.
    gold_texts = []
    for item in gold:
        gold_texts.append(item["text"])

    # Step 2: group the remaining pool items by label (skipping any gold items).
    examples_by_label = {}
    for item in pool:
        if item["text"] in gold_texts:
            continue
        label = item["label"]
        if label not in examples_by_label:
            examples_by_label[label] = []
        examples_by_label[label].append(item)

    # Step 3: for each label, shuffle and take a few examples.
    random_generator = random.Random(seed)
    lines = ["Here are labeled examples:"]
    labels_with_no_examples = []
    for label in labels:
        if label in examples_by_label:
            examples = examples_by_label[label]
        else:
            examples = []
        random_generator.shuffle(examples)
        chosen_examples = examples[:shots_per_class]
        if len(chosen_examples) < shots_per_class:
            labels_with_no_examples.append(label)
        for item in chosen_examples:
            lines.append("Sentence: " + item["text"] + "\nLabel: " + label)

    # Step 4: if the pool could not supply enough spare examples, say so loudly -
    # a few-shot prompt missing whole labels is not the prompt you think it is.
    if len(labels_with_no_examples) > 0:
        print("WARNING: not enough spare pool items for", shots_per_class,
              "example(s) of:", ", ".join(labels_with_no_examples))
        print("         Those labels get fewer examples (or none). This usually means")
        print("         POOL_PATH points at a small DEMO file rather than a full pool.")

    # Step 5: glue the example block in front of the base prompt.
    example_block = "\n\n".join(lines)
    return example_block + "\n\nNow classify this one.\n\n" + base_prompt

**`evaluate`** prints per-class precision/recall/F1, Cohen's κ and the confusion matrix — and **returns the macro-F1 as a number**, which is what lets you collect one per round. Macro-F1 is the plain average of the per-class scores, so every class counts the same however rare it is; that is why a balanced sample and a macro average go together. `ordered=True` adds a weighted κ, which counts a near miss as a smaller error than a far one — use it only if your labels sit on a scale, and pass `labels=LABELS_ORDER` so it knows what that scale is.

In [ ]:
def evaluate(gold: list[dict[str, str]],
             predictions: list[str],
             ordered: bool = False,
             labels: list[str] | None = None,
             title: str = "Confusion matrix") -> float:
    """Score predictions against gold: per-class P/R/F1 + macro, Cohen's kappa, and a
    confusion-matrix heatmap. Returns the macro-F1 as a number.

    ordered=True adds QUADRATIC WEIGHTED kappa — use it only when the labels sit on a
    scale (A1 < A2 < ... < C2), so that a near miss counts as a smaller error than a
    far one. For unordered categories, plain kappa is the one to report.

    IMPORTANT for ordered=True: the scale is taken from `labels`, in the order given.
    Left off, `labels` is read off the gold set and sorted ALPHABETICALLY — which is
    correct for A1..C2 and Move 1..3, but wrong for something like Low/Mid/High
    (alphabetical puts High first). If your labels are ordered and not alphabetical,
    pass them yourself: evaluate(gold, pred, ordered=True, labels=LABELS_ORDER).

    Args:
        gold: the gold items, each with a "label" key.
        predictions: one predicted label per gold item, in the same order.
        ordered: True when the labels sit on a scale.
        labels: the labels to score, in scale order. Left out, they are read off the
            gold set and sorted alphabetically.
        title: the heading to put on the confusion matrix.

    Returns:
        The macro-F1, so you can collect it round by round.

    Example:
        >>> f1_by_round["1 zero-shot"] = evaluate(gold, predictions, ordered=True)
    """
    ### Step 1: line the two label lists up, gold first ###
    y_true = []                          # the correct labels, from the gold set
    for item in gold:
        y_true.append(item["label"])
    y_pred = predictions                 # the model's labels, in the same order

    if labels is None:
        labels = label_set(gold)

    ### Step 2: per-class precision / recall / F1, as a text table ###
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0))

    ### Step 3: one overall number — agreement corrected for chance ###
    # Only meaningful if there is more than one label to be right or wrong about.
    if len(set(labels)) < 2:
        print("Cohen's kappa            undefined (only one label present)")
    else:
        print(f"Cohen's kappa            {cohen_kappa_score(y_true, y_pred):.3f}")
        if ordered:                      # only when the labels sit on a scale
            weighted = cohen_kappa_score(y_true, y_pred, labels=labels,
                                         weights="quadratic")   # near misses hurt less
            print(f"Cohen's kappa (weighted) {weighted:.3f}   <- labels are ordered")
            # Say WHICH order we used, so a wrong one is visible rather than silent.
            print("  scale order used:", " < ".join(labels))

    ### Step 4: draw the same information as a picture ###
    matrix = confusion_matrix(y_true, y_pred, labels=labels)
    plot_confusion_matrix(matrix, labels, title)

    ### Step 5: one number to carry from round to round ###
    macro_f1 = f1_score(y_true, y_pred, labels=labels,
                        average="macro", zero_division=0)
    return macro_f1

Run this to check the definitions above took effect. It prints the first line of the function the notebook will actually use, and the description of each argument.

In [ ]:
help(run_prompt)

### Now send it to the model

This is the slow cell: one API call per dev item, paced a few seconds apart to stay inside the free tier. Forty items takes a couple of minutes.

It is on its own **deliberately**. Scoring and reading the errors are separate cells below, so that looking at your results again costs you nothing. If they shared a cell with this one, every re-read would re-run every call and spend your group's quota a second time.

In [ ]:
pred0 = run_prompt(PROMPT, dev)

### Now score it

`evaluate` prints the table and the matrix, and hands back the macro-F1, which we store under a name we choose.

`ordered=False` is the safe default. Change it to `True` if — and only if — your labels sit on a **scale** (A1 < A2 < … < C2, Low < Mid < High), where a near miss is a smaller error than a far one. Move 1 / Move 2 / Move 3 are *not* a scale: they are three different jobs, not three amounts.

In [ ]:
f1_by_round["round0 baseline (dev)"] = evaluate(dev, pred0,
                                               ordered=False,
                                               labels=LABELS_ORDER)

### Now read what it got wrong

Do not skip this. The errors are the only thing that tells you *what to change*; F1 only tells you afterwards whether the change worked. This is the cell that decides your next round.

In [ ]:
show_errors(dev, pred0)

## Step 3 — Iterate, driven by the errors

Two or three more rounds. The loop is always the same, and the middle step is the one that matters:

```
run  →  score  →  READ THE ERRORS  →  change ONE thing  →  run again
```

Before you touch the prompt, look at the error table from the round you just ran and ask **what these misses have in common**. There are only a few answers, and each points somewhere different:

| What you see in the errors | What it suggests |
|---|---|
| One class swallows everything | the model has not understood that class's boundary — define it, or show an example of it |
| Two labels traded in both directions | the *distinction* is unclear, to the model and possibly to your coders too |
| Lots of `??` | the model is not answering in the format you asked for — fix the instruction, not the definitions |
| Errors scattered with no pattern | you may be at the ceiling of what the prompt can do; consider whether the items are simply hard |

Then change **one** thing, and say beforehand what you expect it to do. "Added examples" is not a reason; *"Move 2 and Move 3 traded in both directions, so I gave it one example of each"* is. Write it down as you go — reconstructing it afterwards from a stack of F1 numbers is much harder than it sounds, and it is report section 2.

A round that made things **worse** is a result, not a mistake. Keep it in the table. It is often the most informative row you have.

`build_fewshot` draws examples from the pool while avoiding anything in your gold set — otherwise you would be testing the model on answers you had just shown it. It replaces typing the examples out by hand, which is how you did it in the Day 3 iterations.

**Save each version as its own prompt file** (`v0`, `v1`, `v2`). A prompt you overwrote is a round you cannot report.

One question to ask after each round: did the confusion matrix change **shape**, or did every cell shift a little? Those two call for different next moves.

In [ ]:
# ══ STEP 3 · Round 1 — build a new prompt ═════════════════════════════════
# Adds worked examples to the prompt you already have, and prints the result
# so you can read it before you spend a round on it.
# Creates: PROMPT_v1

# ✏️ this runs as written — the work is deciding whether it should

# Either add examples to the prompt you already have …
#
# `gold`, not `dev`: it is the list of items NOT to use as examples, and
# that has to include the test half, or the answers leak into the prompt.
#
# Note the NEW NAME on the left. Writing `PROMPT = build_fewshot(PROMPT, ...)`
# would add examples to the prompt that already has them every time you
# re-ran this cell, silently, and you would not see it in the numbers.
PROMPT_v1 = build_fewshot(PROMPT, pool, gold)

# … or write a new prompt file (see the %%writefile example above) and load
# that instead:
# PROMPT_v1 = load_prompt(ROOT / "prompts" / "my_prompt_v1.txt")

print(PROMPT_v1)


### Now run this round

Again on its own, and again for the same reason: one API call per dev item, so a re-run costs your group real quota.

In [ ]:
pred1 = run_prompt(PROMPT_v1, dev)

### Now score it, and read the new errors

Name the key for what you **changed**, not which round it was. In the report, *"round1 four examples per label"* is an argument; *"round1"* is a row number.

Keep `ordered` the same as the baseline, or the two rounds are not comparable.

In [ ]:
f1_by_round["round1 four examples per label (dev)"] = evaluate(
    dev, pred1, ordered=False, labels=LABELS_ORDER)

Now the errors again — these are what design round 2.

In [ ]:
show_errors(dev, pred1)

### Then do it again

For round 2, copy the **three cells above** — change the prompt, run it, score it — and rename `PROMPT_v1` / `pred1` to `PROMPT_v2` / `pred2` as you go, along with the key in `f1_by_round`. Reusing a name is how a round silently scores the round before it and reports an identical F1.

Whichever version wins on dev is the one you carry into step 4.

## Step 4 — The held-out run

This is the first time all week that `TEST_PATH` gets opened. Your prompt is settled, you run it once against items it has never been tuned against, and **whatever comes out is what you report.**

Expect it to be lower than your best dev round. That is the normal result, not a failure and not a sign you did something wrong: the gap is roughly how much of your improvement was tuning to those particular dev items rather than to the task. Reporting the gap is a stronger finding than reporting a high number, and notebook 05 asks you for it.

A hosted model is only *best-effort* reproducible even at `temperature=0`, so the run is frozen to a file and every number in notebook 05 comes out of that file rather than out of this session's memory.

**Nothing here stops you running it twice.** Stopping you would be the wrong design — a genuine mistake at four o'clock on the last day needs a way forward. So instead: nothing is overwritten (a second run lands in `..._predictions_attempt2.json`), and every scoring appends a line to a log that goes into your submission. A second attempt is allowed. It is just not invisible, and §5 of your report has to account for it.

**One person runs this.** It is the run you will be defending.

In [ ]:
# ══ STEP 4 · Open the test set and pick your prompt ═══════════════════════
# Opens the held-out items — the only cell all week that does — and notes your
# best dev score before the test row joins the table.
# Creates: test, FINAL_PROMPT, best_dev

# ✏️ this runs as written — the work is deciding whether it should

test = load_gold(TEST_PATH)      # the first time this file is opened all week

# The prompt that won on dev. Change this to PROMPT_v2, or whichever round
# came out best — it is the one you are about to be judged on.
FINAL_PROMPT = PROMPT_v1

# Your best DEV round, noted BEFORE the test row joins the table. The gap
# between this and what you are about to get is a finding in its own right.
best_dev = max(f1_by_round.values())
print("best dev round:", round(best_dev, 3))


### Now the run itself

**One cell, one call, and it is the one you will be defending.** It is on its own so that nothing else in this notebook can fail *after* it and leave you re-running it — every re-run is another line in the log and another attempt number to account for in report §5.

`freeze_test_run` does five things, and it does them in one cell precisely because stopping halfway through them is the failure worth preventing — a log that disagrees with the predictions file is worse than either alone. Watch them go past as it runs:

1. Runs `FINAL_PROMPT` over the test items.
2. **Saves before scoring anything.** It never overwrites: a second run lands in `..._predictions_attempt2.json` beside the first, and both stay.
3. Reads that file straight back off disk and scores *those* predictions, which checks that the file you will quote in your report is the file you think it is.
4. Appends one line to the log that travels in your submission bundle: the score, which attempt it was, and a fingerprint of the prompt that produced it.
5. Adds the test score to `f1_by_round` **last**, so the table in your report reads as the dev rounds in order with the held-out score at the bottom, and saves that table for notebook 05.

The four file names it is handed are the four files it touches. `note=` is the only argument that is a decision: a second attempt with the **same** prompt is that prompt run twice; a second attempt with a **different** one is a prompt tuned after seeing the held-out set. The log fingerprints the prompt either way, so say which it was.

In [ ]:
macro_f1 = freeze_test_run(FINAL_PROMPT, test, f1_by_round,
                           PRED_PATH, TESTLOG_PATH, ROUNDS_PATH, PROMPT_FILE,
                           dev_f1=best_dev,
                           ordered=False,   # True only if your labels are a SCALE
                           labels=LABELS_ORDER,
                           note="")   # ← if this is not attempt 1, say WHY here

---

**Next:** open `05_report.ipynb`. It loads the files you just wrote and nothing else — so from here on, your numbers cannot move.